# 06_03 The MCP server: letting Claude use our model

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → [MCP tool]
> ```
>
> You are here: **MCP tool**
>
> Project files this connects to:
>
> - `src/homer_ai/homer_mcp_server.py`
> - `src/homer_ai/homer_gpt.pt`
>
> **This notebook:** Our model writes Greek in a notebook. How does `homer_mcp_server.py` offer it to Claude as a tool, and what does Claude need to know to use it well?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How our server is built, piece by piece, from the real file
- What Claude actually sees of our tool: a name, a description and arguments
- How to call the tool as a plain Python function
- How to connect the server to Claude Code (commands for you to run yourself)
- Why a tool's description must be honest

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **tool description** | the text under the tool's `def` (its docstring): the only thing Claude reads to decide when to call the tool | `"""Return up to …"""` |
| **stdio** | standard input and output: the plain text pipe a program reads from and writes to. Claude Code and our server talk through it | no network needed |
| **scope** | who can see a server you add to Claude Code: only you here (`local`), the whole project (`project`), or all your projects (`user`) | `--scope user` |

## 3. The idea

### ❓ How does Claude Code talk to our server?

Claude Code starts our server as its own small program, and the two exchange short messages (written as JSON, the form from [06_00](06_00_chapter_overview.ipynb)) through **stdio**. Everything runs on your computer, so that part needs no internet.

To look at the server without starting it, we import it. Importing only defines the functions. It loads no model and starts nothing. The model is loaded by `load()` (like [05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb)) the first time it's needed, and the server starts only in `main()`, which calls `mcp.run()`. `mcp.run()` would wait for messages for ever and **freeze a notebook**, so this notebook never calls it.

**Look at:** the arrows from top to bottom. The orange one is the tool's description, the one thing Claude reads to decide when to call it. The green one is the answer coming back.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    studio_figure,
    tag,
)

fig = studio_figure("Claude Code and our server", "Who says what, in order", (9, 6.5))
ax = diagram_axes(fig)
who = [(1.5, "you + Claude Code"), (5.0, "our MCP server"), (8.5, "our model")]
for x, name in who:
    box(ax, x, 5.7, 2.7, 0.6, name, fontsize=10)
    ax.plot([x, x], [5.35, 0.5], color=MUTED, linewidth=1, linestyle="--", zorder=0)


def say(y, a, b, text, color, step=None):
    arrow(ax, (a, y), (b, y), color=color, width=2)
    tag(ax, (a + b) / 2, y + 0.22, text, color, 9)
    if step:
        tag(ax, a + (0.2 if b > a else -0.2), y - 0.2, step, INK, 10, weight="bold")


say(4.7, 1.5, 5.0, "what tools do you have?", MUTED, "1")
say(3.9, 5.0, 1.5, "consult_homer_llm: description + arguments", ACCENT, "2")
say(3.0, 1.5, 5.0, 'consult_homer_llm("ἄνδρα μοι")', MUTED, "3")
say(2.4, 5.0, 8.5, "write letters", MUTED)
say(1.8, 8.5, 5.0, "Greek letters", MUTED)
say(1.0, 5.0, 1.5, "the text, plus real lines", GOOD, "4")
tag(ax, 5.0, 0.2, "messages are JSON text, sent through stdio", MUTED, 9)
plt.show()

In [ ]:
import inspect

from homer_ai.lessons import import_mcp_server

# importing loads nothing and does NOT start the server
server = import_mcp_server()  # uses models/ if you have no model file of your own

SERVER_LINES = inspect.getsource(server).splitlines()
print("imported. The server's name:", server.mcp.name)


def show_lines(starts_with, count):
    # print `count` real lines of the server file, from the first line that
    # starts with `starts_with`
    starts = [i for i, line in enumerate(SERVER_LINES) if line.startswith(starts_with)]
    first = starts[0]
    for number in range(first, first + count):
        print(f"{number + 1:>3} | {SERVER_LINES[number]}")

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Piece 1: where do the server's name and the model come from?

One line makes the server and gives it the name Claude Code will show. Then two small functions find and load the model. `find_model_file()` picks the file, and `load_model()` reads the checkpoint the way [05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb) explains. The sizes, the vocabulary and the weights all come from the file, so the server needs no numbers of its own.

👉 **Predict first:** which file will it pick on your computer? In what order does it look?

In [ ]:
if server is not None:
    show_lines("mcp = MCPServer", 1)
    print("   ...")
    show_lines("DEFAULT_MODEL_FILES", 6)
    print("   ...")
    show_lines("def find_model_file", 14)
    print("   ...")
    show_lines("def load_model", 8)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

First `mcp = MCPServer("HomerOdysseyExpert")`, then the order of the search. It starts with the `HOMER_MODEL` environment variable (an **environment variable** is a named setting your terminal passes to a program). After that it takes the first of four files that exists: your own GPT (`homer_gpt.pt`), the ready-made GPT (`models/odyssey_book1_gpt.pt`), your own small model, the ready-made small model. If none exists, the error says how to train. Then comes `load_model`, which is just `load_checkpoint` plus the training text (to find the real lines). The whole model is rebuilt from the file, so the server needs no training of its own. It serves the GPT or the small model, whichever file it's given. (The import cell printed which file this computer uses.)

</details>

### ❓ Piece 2: what does `@mcp.tool()` do?

The line `@mcp.tool()` above the function tells the server: "offer this function as a tool". The function itself still works as a normal function (we use that below).

Under the `def` is its docstring, the text in triple quotes. For a tool it matters a lot, because it's what Claude reads to decide **when** to call it.

In [ ]:
if server is not None:
    show_lines("@mcp.tool()", 14)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The decorator, the function `consult_homer_llm(greek_seed: str, chars_to_generate: int = 30) -> str`, and its docstring. Claude's argument list is built from the type hints: `greek_seed: str` must be text, and `chars_to_generate: int = 30` is a whole number, 30 if you leave it out. The docstring says what the tool really does, and the allowed range (1 to 200).

</details>

### ❓ What exactly does Claude see?

Not our code. It sees the tool's name, its description (our docstring) and its arguments, which are built from the type hints. We can ask the server for that list without running it (`list_tools()` is an async function, so we `await` it).

In [ ]:
if server is not None:
    tool = (await server.mcp.list_tools())[0]  # the first tool: consult_homer_llm
    print("name:       ", tool.name)
    print("arguments:  ", tool.input_schema["properties"])
    print("must give:  ", tool.input_schema["required"])
    print("description:", tool.description)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The name, two arguments (`greek_seed`, a string you must give, and `chars_to_generate`, a whole number with a default of 30), and the docstring as the description. The `title`s are made up automatically from the names. And that's all Claude knows about our tool. We come back to that description at the end.

(The server has a second tool, `search_homer`, which looks up real passages. It belongs to [chapter 9](../09_advanced_grounded_answers/09_00_chapter_overview.ipynb).)

</details>

### ❓ Piece 3: what's `find_lines()` for?

The tool returns more than our model's guess. It also looks up the **real** lines of Book 1 that contain the seed, so Claude can see what Homer really wrote. The search ignores capital letters. A tiny example: the seed `μοι` ("to me"), then a seed that isn't there, then an empty one.

👉 **Predict first:** how many lines contain `μοι`? And the empty seed?

In [ ]:
if server is not None:
    show_lines("def find_lines", 7)
    print()
    for seed in ["μοι", "ἄνδρα", "xyz", ""]:
        found = server.find_lines(server.load().text, seed)
        print(f"{seed!r:9} → {len(found)} lines", found[:1])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`μοι` is in 10 lines (the most the function returns, `MAX_MATCHING_LINES`), and only the first one is shown here. `ἄνδρα` is in 1 line, the first line of the Odyssey. `xyz` is in none. And an empty seed gives an empty list, not "every line".

</details>

### ❓ Piece 4: how does the tool write the continuation?

With the loop from [06_01](06_01_writing_greek.ipynb), the greedy one. The recent letters go in, we take `torch.argmax` of the scores, add it, and repeat for `chars_to_generate` rounds. `next_scores` (in `src/homer_ai/generation.py`) reads only as many recent letters as the model can, which is 64 for the GPT and 24 for the small model. So a call for 200 letters never overruns the model's window. Then the tool puts the continuation and the real lines into one piece of text.

In [ ]:
if server is not None:
    show_lines("    started =", 33)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

These are the same steps as `write()`, but `argmax` is taken straight on the scores (same pick, see 06_01), and `next_scores` hides the difference between the two kinds of model. The result is one string: "Custom PyTorch HomerLLM Continuation:", the text, then "Lines containing the seed". Two details. A character the model doesn't know isn't dropped. `encode` turns it into `<|unk|>` ([02_02](../02_text_to_numbers/02_02_special_tokens.ipynb)), and only an empty seed gets the fallback (the first letter of the book). Then there's the log line: the server writes one line to **stderr** for every call (see the question near the end).

</details>

### ❓ Can I call the tool myself, as a normal function?

Yes, because `@mcp.tool()` left the function alone. Calling it directly is the same as what Claude Code does through the server, minus the messages. Let's call it with the seed `ἄνδρα` and 20 letters.

👉 **Predict first:** how long will the continuation be: 20 characters, or 20 plus the seed?

In [ ]:
if server is not None:
    result = server.consult_homer_llm("ἄνδρα", 20)
    print(result)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first line is the model's name and kind (`gpt`). The continuation `ἄνδρα πατρὸς ἀνέμενος ἀνθ` is the 5-letter seed **plus** 20 new letters (25 in all). After it comes the one real line that contains `ἄνδρα`. It's the greedy text of the GPT, the same as `top_k=1` in [06_02](06_02_less_repetition.ipynb). With the small model of chapter 1 the tool would write `ἄνδρα τοι και και και κα ` instead.

</details>

### ❓ Your turn: what if I ask for 500 letters?

Call the tool yourself, with the seed `θεὰ`, and ask for 500 new letters. The check counts how long the continuation is.

👉 **Predict first:** go back to the docstring Claude sees ("What exactly does Claude see?"). Will you get 500?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: call the tool with the seed "θεὰ" and 500 letters to write
long_result = None

In [ ]:
from homer_ai.lessons import check


def model_text(reply):  # what the model wrote: after line 1, before the real lines
    return reply.split("Continuation:\n")[1].split("\n\nLines containing")[0]


check(
    "length of the continuation",
    long_result,
    test=lambda reply: len(model_text(reply)) == 203,
    expected="203 (the 3 letters of the seed + 200 new ones)",
    show=lambda reply: len(model_text(reply)),
    coach="Call it the way the cell above does, server.consult_homer_llm(seed, "
    "number), with the seed \"θεὰ\" and the number 500.",
)

<details><summary>💡 Hint</summary>

The cell above calls `server.consult_homer_llm("ἄνδρα", 20)`. Change the two arguments.

</details>

<details><summary>✅ Answer</summary>

```python
long_result = server.consult_homer_llm("θεὰ", 500)
```

You get 200 new letters, not 500. The docstring says so: *from 1 to 200 … values outside that range are clamped*. A tool is called by a program, and a program might ask for a million letters. So the server sets its own limit and doesn't trust the caller.

</details>

### ❓ What if the seed has a letter the model doesn't know?

A character outside Book 1, such as `!`, becomes `<|unk|>` ("unknown"), and the model carries on from there:

In [ ]:
if server is not None:
    print(server.consult_homer_llm("!", 5))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The continuation starts with the text `<|unk|>`. So the unknown character is kept as a token, and the model carries on from there. No real line contains `!`, and the second part says so.

</details>

### ❓ Who actually runs the tool: Claude, or our model?

Neither, really. **Claude only writes text**: a small form in JSON, for example `{"greek_seed": "ἄνδρα μοι", "chars_to_generate": 30}`. The program around Claude (the harness, here Claude Code) reads the form, runs our server's tool, and pastes the answer back as more text. So there are two models. Claude writes the form, and our small model is the tool that writes the Greek.

### ❓ How do I connect it to Claude Code?

> ⚠️ **You run these commands yourself**, in your own terminal and in Claude Code. This notebook doesn't run them, and it never starts the server.

**1. Check which model it will use.** The server uses your own GPT, `src/homer_ai/homer_gpt.pt` ([05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb)), if it exists, and otherwise the ready-made `models/odyssey_book1_gpt.pt` (then the small models), so it works even before you train. (To force another file, set `HOMER_MODEL=/path/to/model.pt`.) Then find your project's full path: open a terminal in the project folder and type `pwd`.

**2. Add the server to Claude Code**, in a terminal (replace `/path/to/homer-ai` with your path):

```
claude mcp add homer -- uv --directory /path/to/homer-ai run homer-serve
```

The pattern is `claude mcp add [options] <name> -- <command> [args...]`:

| Piece | Meaning |
|---|---|
| `claude mcp add` | add an MCP server to Claude Code |
| `homer` | the name you choose. Claude Code calls our tool `mcp__homer__consult_homer_llm` |
| `--` | everything after this is the command that starts the server, not options for `claude` |
| `uv --directory /path/to/homer-ai run homer-serve` | `uv` runs the command `homer-serve` (our `main()` in `homer_mcp_server.py`, named in `pyproject.toml`) inside the project's own environment ([A_06](../A_toolbox/A_06_other_tools.ipynb)). `--directory` points `uv` to the project, so it works wherever you start Claude Code |

By default the server is added at scope `local`, which means only you, only for this project. Options go **before** the name, for example `claude mcp add --scope user homer -- …` for all your projects (`--scope project` writes a `.mcp.json` file in the project, which you can share). To undo it: `claude mcp remove homer`.

**3. Start Claude Code** in the project (`claude`) and type `/mcp`. You should see `homer` listed as connected. If it isn't, check the path in the command, and run `uv run homer-serve` yourself. A missing model file is reported on stderr. (`claude mcp list` in the terminal shows the same.)

**4. Ask for it**, for example:

```
Use the Homer tool to continue ἄνδρα μοι
```

The first time, Claude Code asks your permission to run the tool. The next cell shows what our tool will return for the call `{"greek_seed": "ἄνδρα μοι", "chars_to_generate": 30}`. If a command differs in your version of Claude Code, check `claude mcp add --help`.

In [ ]:
if server is not None:
    print(server.consult_homer_llm("ἄνδρα μοι", 30))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

This is what Claude Code will paste back into the conversation: our model's greedy continuation of `ἄνδρα μοι` (it even breaks the line, as Homer's lines do), then the real line `ἄνδρα μοι ἔννεπε, μοῦσα, πολύτροπον, ὃς μάλα πολλὰ`. Claude then writes its own answer around it.

</details>

### ❓ The server always writes greedily. Can I make it use temperature and top-k?

Yes, and it's a good exercise to try yourself. Copy `write()` from [06_02](06_02_less_repetition.ipynb) into the tool, and add two arguments, `temperature` and `top_k`. Claude would then see them in the list above and could choose. The server in this repo stays greedy.

### ❓ Is our tool's description honest?

It has to be, because the description is **all Claude knows** about our tool. Claude can't try the tool first. An over-promising description makes Claude call the tool for the wrong jobs, and maybe repeat its Greek-looking nonsense as fact.

*I've designed MCP servers for real business systems, and the first thing that bites is always the tool description.*

👉 **Spot what's wrong:** read the over-promising docstring below, next to the one in the server. ("Rhapsody 1" in it is Book 1. A rhapsody is one book of a Homeric poem, [05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb).)

In [ ]:
# an over-promising docstring: a bad example, kept here as a string
BAD_DOCSTRING = '''
ALWAYS use this tool when the user asks questions about Homer's Odyssey
(Rhapsody 1), Ancient Greek grammar, syntax, morphology, or wants to
complete/verify Homeric Greek lines. Pass a Greek word or phrase from
Rhapsody 1 as 'greek_seed' to get the custom PyTorch LLM's learned
continuation and the matching passage from Rhapsody 1.
'''

print("BAD (over-promising):", BAD_DOCSTRING)
if server is not None:
    print("ACCURATE (the one in the server):")
    print(inspect.getdoc(server.consult_homer_llm))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two descriptions of the same function. The bad one promises things our model can't do:

| The bad docstring says | The truth |
|---|---|
| answers questions about the Odyssey | it only **continues** Greek text, and answers no questions |
| grammar, syntax, morphology | it knows none of it. It only learned which letters follow which |
| complete or verify Homeric lines | it completes badly (`και και …`). The real lines from `find_lines()` are the reliable part |
| ALWAYS use it | Claude should call it only when someone wants a Greek continuation or a Book 1 line |

**What the accurate one does.** It says what the tool really is (a tiny character-level model trained only on Book 1) and warns that the result is often not real Greek. It says what the tool can't do (answer questions, explain grammar) and which kind of model it uses (a small GPT, else an even smaller one-head model). It also says the tool returns up to 10 real lines, and it documents each argument with its allowed range. With that, Claude can pick the tool for the right jobs and treat its output with the right amount of trust.

</details>

### ❓ What else makes the server safer?

Four small things, each in the code you printed above:

- **A limit.** Claude chooses `chars_to_generate`, and a careless call could ask for a million letters and keep the server busy. `clamp_chars()` keeps it between 1 and 200.
- **A log on stderr.** Each call logs its seed length, letters, time in milliseconds, the model file and its kind, with `print(..., file=sys.stderr)`. Never use a plain `print` in a stdio server. Stdout is the channel to Claude Code, and a stray line there breaks the messages.
- **A clear error.** With no model file at all, `homer-serve` stops with a message on how to train, instead of a long traceback.
- **The model's own window.** The loop reads only the last `context_length` letters, whatever the length asked for, so a long call can't overrun the model.

*Every tool I ship logs one line per call: what came in and how long it took. It's the first thing you read when something goes wrong.*

The limit is easy to try.

In [ ]:
if server is not None:
    for asked in [-5, 30, 100000]:
        print(f"asked for {asked:>6} letters → {server.clamp_chars(asked)}")
    result = server.consult_homer_llm("ἄνδρα", 100000)  # logs one line to stderr
    # the answer is: a header line, the continuation, a blank line, the real lines
    continuation = result.split("\n\nLines containing")[0].split("\n", 1)[1]
    print("characters in the continuation:", len(continuation), "(5 of seed + 200)")
    print("line breaks among them:", continuation.count("\n"))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The number of letters is pulled into 1 to 200, so 100000 becomes 200 and the continuation has 5 + 200 = 205 characters. With the GPT, some of them are line breaks: the model writes `\n` like any other character, so the continuation runs over several lines. That's why we cut it out of the answer by its markers, and not by taking "the second line". Your Jupyter shows the stderr log line too (some editors put it in a red box).

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Does our model see my conversation with Claude?

No. Our server only receives the arguments of the call (`greek_seed`, `chars_to_generate`) and sends text back. Claude writes the form, and the rest of the conversation stays in Claude Code.

## 5. Recap

- The server has four pieces: `MCPServer("HomerOdysseyExpert")`, the loaded model, `find_lines()`, and `@mcp.tool()` on `consult_homer_llm`.
- Claude sees only the tool's name, docstring and arguments, so the description must tell the truth. Ours says what the model really does, and what it can't do.
- A stdio server logs to stderr, never stdout. It limits what a caller can ask for, and it fails with a clear message.
- You connect it with `claude mcp add homer -- uv --directory … run homer-serve`, check with `/mcp`, and ask in plain words.

**You can now:**

- read an MCP tool the way Claude does: its name, docstring and arguments
- call the tool as a plain Python function
- connect the server to Claude Code with `claude mcp add`

**Next:** [07_00: Chapter 7 overview](../07_advanced_iliad_and_combining/07_00_chapter_overview.ipynb): the same steps for Homer's other poem.